# Test Baseline Model (mit Explode)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/test_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

# Defining Model


In [4]:
#model = 'XGBoost'
model = 'RandomForest'
#model = 'TabPFN'

# Add Physical Columns Pullout and Interfacial_Failure


In [5]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

x_train['Interfacial_Failure'] = compute_interfacial_failure(x_train) 
x_train['Pullout_Failure'] = compute_pullout_failure(x_train) 
x_dev['Interfacial_Failure'] = compute_interfacial_failure(x_dev)
x_dev['Pullout_Failure'] = compute_pullout_failure(x_dev)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(X=x_dev)

# Fit Model

In [6]:
if model == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 1,
        'eta': 0.57,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 20
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 19,
            'max_depth': 5,
            'min_samples_split': 6,
            'min_samples_leaf': 3,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)


# Check Validation Data

In [7]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category - Data-Driven Training with IFPL Features"
)

# Check Validation Loss and R2

In [8]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = root_mean_squared_error(y_dev, predictions)
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  165.33
RMSE: 273.83
R2: 0.73
